# Entregable 4 · Pipeline ETL — Notebook de ejemplos
## Dashboard de Bajas de Activos de TI — UAO

**Complejidad:** Alta · **Formato:** Script Python (paquete `etl/`) + este Notebook.

Este notebook demuestra, fase por fase, cómo funciona el pipeline ETL que alimenta el dashboard. El código productivo vive en el paquete `etl/` (módulos `extract`, `transform`, `validate`, `load`, `run_pipeline`). Aquí se ilustra su uso con ejemplos ejecutables.

## 0. Preparar el entorno
Añadimos el paquete `etl/` al path para poder importar sus módulos.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../etl'))
from logger import get_logger
logger = get_logger('demo')
print('Módulos del pipeline disponibles')

Módulos del pipeline disponibles

## 1. Extract — cargar datos de las fuentes
La fase Extract lee los CSV de origen y valida el contrato de esquema.

In [ ]:
from extract import extraer
crudo, meta = extraer(logger)
print('Filas crudas:', meta['total'])
print('Por archivo:', meta['por_archivo'])
crudo[0]

== FASE EXTRACT ==
Extraídas 473 filas de 'bajas_2025.csv'
Extraídas 451 filas de 'bajas_2026.csv'
Filas crudas: 924
Por archivo: {'bajas_2025.csv': 473, 'bajas_2026.csv': 451}

## 2. Transform — limpieza, derivación y cálculo de KPIs
Normaliza valores, deriva Categoría/Marca/tiempo e indicadores, y calcula los 14 KPIs.

In [ ]:
from transform import transformar, calcular_kpis
limpio, met = transformar(crudo, logger)
kpis = calcular_kpis(limpio, logger)
print('Unidades:', kpis['KPI-01_total_unidades'])
print('Informes:', kpis['KPI-02_total_informes'])
print('Categoría top:', kpis['KPI-05_categoria_top'], kpis['KPI-05_pct_categoria_top'], '%')
print('Variación anual:', kpis['KPI-09_variacion_anual_pct'], '%')

== FASE TRANSFORM ==
Transformadas 924 filas | fechas inválidas: 0
== CÁLCULO DE KPIs ==
Unidades: 1322
Informes: 104
Categoría top: Mouse 20.7 %
Variación anual: 3.1 %

**Ejemplo de registro transformado** (nótese las columnas derivadas `Categoria`, `Marca`, `Trimestre`, `TieneSerial`, etc.):

In [ ]:
limpio[0]

{'Archivo de Origen': 'Bajas 2025.pdf', 'Fecha': '2025-01-16', 'Anio': 2025,
 'Mes': 1, 'NombreMes': 'Enero', 'AnioMes': '2025-01', 'Trimestre': '2025-T1',
 'Categoria': 'Telefonia IP/Fija', 'Marca': 'Sin marca/Generico',
 'TieneSerial': 'Si', 'TieneInventario': 'Si', 'EsLote': 'No', ...}

## 3. Validación — checks de integridad
Siete checks; los errores abortan el pipeline, las advertencias solo se registran.

In [ ]:
from validate import validar
res = validar(limpio, met, logger)
print('¿Validación OK?', res.ok())
for nombre, estado, detalle in res.checks:
    print(f'  [{estado}] {nombre}: {detalle}')

== FASE VALIDACIÓN ==
  [PASS] Volumen mínimo: 924 filas (mín 100)
  [PASS] Esquema de salida: 18 columnas requeridas
  [PASS] Fechas válidas: 0.0% inválidas (máx 5.0%)
  [PASS] Cantidades positivas: 0 con cantidad <= 0
  [PASS] Años en rango: 0 fuera de [2025, 2026]
  [PASS] Consecutivo presente: 0 sin consecutivo
  [PASS] Trazabilidad inventario: 40.4% sin inventario (informativo)
¿Validación OK? True

## 4. Load — guardar datos limpios y KPIs
Escribe la tabla de hechos (CSV UTF-8 para Power BI) y los KPIs (JSON) de forma atómica.

In [ ]:
from load import cargar
destino = cargar(limpio, kpis, logger)
print(destino)

== FASE LOAD ==
Tabla de hechos escrita: .../bajas_consolidado.csv (924 filas)
KPIs escritos: .../kpis.json (16 métricas)
{'fact': '.../bajas_consolidado.csv', 'kpis': '.../kpis.json', 'filas': 924}

## 5. Logs — trazabilidad de la ejecución
Cada corrida genera un archivo en `etl/logs/` con marca de tiempo. Útil para auditar.

In [ ]:
print('Bitácora de esta ejecución:', logger.log_path)
# En producción se revisa el último log para confirmar el resultado

Bitácora de esta ejecución: ../etl/logs/etl_YYYYMMDD_HHMMSS.log

## 6. Ejecución completa del pipeline
En la práctica no se llaman las fases una a una, sino el orquestador, que las encadena con manejo de errores y devuelve un código de salida.

In [ ]:
# Desde la terminal:
#   cd etl && python3 run_pipeline.py            # ejecución normal
#   cd etl && python3 run_pipeline.py --dry-run  # valida sin escribir
from run_pipeline import main
codigo = main(dry_run=True)   # demo sin escritura
print('Código de salida:', codigo, '(0 = éxito)')

... PIPELINE COMPLETADO ...
Código de salida: 0 (0 = éxito)

## 7. Scheduler (opcional) — automatización
El pipeline es idempotente, por lo que puede programarse sin riesgo. Ejemplos en `etl/scheduler/`:

- **cron** (Linux/macOS): `0 6 1 * * cd .../etl && python3 run_pipeline.py`
- **Programador de tareas** (Windows): `schtasks /SC MONTHLY ...`
- **GitHub Actions**: `etl-schedule.yml` (ejecución mensual en la nube).

### Conclusión
El pipeline automatiza por completo la alimentación del dashboard: extrae de los CSV, limpia y enriquece, valida la integridad, calcula los 14 KPIs y publica la tabla de hechos lista para Power BI, dejando trazabilidad en cada ejecución.